# Invoke Amazon SageMaker Endpoints Using the OpenAI SDK

This notebook demonstrates how to deploy and invoke Amazon SageMaker real-time inference endpoints using the OpenAI Python SDK. With OpenAI-compatible API support, SageMaker endpoints accept requests in the [Chat Completions](https://platform.openai.com/docs/api-reference/chat) format — the same interface used by the OpenAI SDK, LangChain, Strands Agents, and other popular AI frameworks.

This notebook covers:
- Generating bearer tokens for authentication using the SageMaker Python SDK
- Deploying and invoking a **single model endpoint**
- Deploying and invoking an **inference component** on a shared endpoint
- Building a multi-agent workflow with **Strands Agents**

**Estimated time:** 20–30 minutes (including endpoint provisioning)  
**Estimated cost:** ~$5–10 USD (ml.g6.12xlarge for <1 hour)

## Prerequisites

Before running this notebook, ensure you have:

1. An AWS account with permissions to create SageMaker endpoints
2. An IAM execution role with the `AmazonSageMakerFullAccess` policy attached
3. An IAM role or user for invocation with the `sagemaker:InvokeEndpoint` and `sagemaker:CallWithBearerToken` permissions
4. Model weights uploaded to Amazon S3 (this notebook uses [Qwen3-4B](https://huggingface.co/Qwen/Qwen3-4B))

> **Note:** The `AmazonSageMakerFullAccess` policy grants S3 access only to buckets with `sagemaker` in the name. If your model is stored in a differently named bucket, add an explicit S3 read policy to your execution role.

## Install dependencies

Install the SageMaker Python SDK (which includes the token generator), the OpenAI SDK, and Strands Agents.

In [ ]:
!pip install --upgrade --quiet sagemaker openai httpx strands-agents strands-agents-tools

## Configuration

Update the following values for your AWS account and model location.

In [ ]:
import boto3
import sagemaker
import time
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.helper.session_helper import get_execution_role
# AWS configuration
REGION = "us-west-2"

# Automatically resolve account ID and default SageMaker execution role
session = Session(boto_session=boto3.Session(region_name=REGION))
ACCOUNT_ID = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
EXECUTION_ROLE = get_execution_role(sagemaker_session=session)

# HF Model ID
MODEL_HF_ID = "Qwen/Qwen3-4B"
# SageMaker vLLM Deep Learning Container
VLLM_IMAGE = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/vllm:0.20.2-gpu-py312-cu130-ubuntu22.04-sagemaker"

# Instance type (1x NVIDIA L4 GPU)
INSTANCE_TYPE = "ml.g6.2xlarge"

sagemaker_client = boto3.client("sagemaker", region_name=REGION)

print(f"Region:         {REGION}")
print(f"Account ID:     {ACCOUNT_ID}")
print(f"Execution role: {EXECUTION_ROLE}")
print(f"Model HF ID:   {MODEL_HF_ID}")

## Bearer token authentication

SageMaker OpenAI-compatible endpoints authenticate using bearer tokens generated from your existing AWS credentials. The token generator is included in the SageMaker Python SDK — no separate package installation is required.

Each token is valid for up to 12 hours. For this notebook, we generate a fresh token before each invocation. For production applications with long-running processes, see the auto-refreshing pattern in [Section 5](#5-auto-refreshing-tokens-for-production).

In [ ]:
from sagemaker.core.token_generator import generate_token
from datetime import timedelta

token = generate_token(region=REGION, expiry=timedelta(minutes=5))
print(f"Token generated successfully")
print(f"Token length: {len(token)} characters")

---

## Part 1: Single model endpoint

A single model endpoint hosts one model and routes all traffic to it. This is the simplest deployment pattern — ideal when you have a dedicated model for a single use case.

### 1.1 Create the endpoint

The following cells create a SageMaker model, endpoint configuration, and endpoint using the vLLM Deep Learning Container with Qwen3-4B.

In [ ]:
import time

# Unique suffix for this run
TIMESTAMP = str(int(time.time()))

SME_MODEL_NAME = f"openai-compat-sme-model-{TIMESTAMP}"
SME_ENDPOINT_CONFIG_NAME = f"openai-compat-sme-epc-{TIMESTAMP}"
SME_ENDPOINT_NAME = f"openai-compat-sme-ep-{TIMESTAMP}"

print(f"Timestamp suffix: {TIMESTAMP}")
print(f"Model:           {SME_MODEL_NAME}")
print(f"Endpoint config: {SME_ENDPOINT_CONFIG_NAME}")
print(f"Endpoint:        {SME_ENDPOINT_NAME}")

# Create the SageMaker model
sagemaker_client.create_model(
    ModelName=SME_MODEL_NAME,
    ExecutionRoleArn=EXECUTION_ROLE,
    PrimaryContainer={
        "Image": VLLM_IMAGE,
        "Environment": {
            "HF_MODEL_ID": MODEL_HF_ID,
            "SM_VLLM_TENSOR_PARALLEL_SIZE": "1",
            "SM_VLLM_MAX_NUM_SEQS": "4",
            "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
            "SM_VLLM_TOOL_CALL_PARSER": "hermes",
            "SAGEMAKER_ENABLE_LOAD_AWARE": "1",
        },
    },
)
print(f"\nModel created: {SME_MODEL_NAME}")

In [ ]:
# Create the endpoint configuration
sagemaker_client.create_endpoint_config(
    EndpointConfigName=SME_ENDPOINT_CONFIG_NAME,
    ProductionVariants=[
        {
            "VariantName": "variant1",
            "ModelName": SME_MODEL_NAME,
            "InstanceType": INSTANCE_TYPE,
            "InitialInstanceCount": 1
        }


    ],
)
print(f"Endpoint configuration created: {SME_ENDPOINT_CONFIG_NAME}")

In [ ]:
# Create the endpoint and wait for it to reach InService status
sagemaker_client.create_endpoint(
    EndpointName=SME_ENDPOINT_NAME,
    EndpointConfigName=SME_ENDPOINT_CONFIG_NAME,
)
print(f"Endpoint creation initiated: {SME_ENDPOINT_NAME}")
print("Waiting for endpoint to reach InService status (this takes 5–10 minutes)...")

waiter = sagemaker_client.get_waiter("endpoint_in_service")
waiter.wait(
    EndpointName=SME_ENDPOINT_NAME,
    WaiterConfig={"Delay": 30, "MaxAttempts": 40},
)
print(f"Endpoint is InService: {SME_ENDPOINT_NAME}")

### 1.2 Invoke the endpoint with the OpenAI SDK

With the endpoint in service, you can invoke it using the OpenAI Python SDK. Set `base_url` to your SageMaker endpoint URL and pass the bearer token as `api_key`.

The URL format for a single model endpoint is:
```
https://runtime.sagemaker.<REGION>.amazonaws.com/endpoints/<ENDPOINT_NAME>/openai/v1
```

The `model` field is passed through to the container as-is. Since routing is determined by the endpoint name in the URL, you can leave this field empty.

Make sure the role you are using to invoke the endpoint has the following permissions:


```
{
      "Version": "2012-10-17",
      "Statement": [
          {
              "Effect": "Allow",
              "Action": "sagemaker:InvokeEndpoint",
              "Resource": "arn:aws:sagemaker:<REGION>:<ACCOUNT_ID>:endpoint/<ENDPOINT_NAME>"
          },
          {
              "Effect": "Allow",
              "Action": "sagemaker:CallWithBearerToken",
              "Resource": "*"
          }
      ]
  }

```

In [ ]:
from openai import OpenAI
from sagemaker.core.token_generator import generate_token
sme_base_url = f"https://runtime.sagemaker.{REGION}.amazonaws.com/endpoints/{SME_ENDPOINT_NAME}/openai/v1"
client = OpenAI(
    base_url=sme_base_url,
    api_key=generate_token(region=REGION)
)

print(f"Base URL: {sme_base_url}\n")


#### Streaming response

Streaming returns partial responses as Server-Sent Events (SSE), allowing you to display tokens as they are generated.

In [ ]:
stream = client.chat.completions.create(
    model="",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain how transformers work in machine learning, in three sentences."},
    ],
    stream=True,
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="")
print()  # newline after stream completes

#### Non-streaming response

A non-streaming request returns the complete response in a single payload, including token usage metadata.

In [ ]:
response = client.chat.completions.create(
    model="",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France? Reply in one sentence."},
    ],
    stream=False,
)

print(f"Response:      {response.choices[0].message.content}")
print(f"Finish reason: {response.choices[0].finish_reason}")
print(f"Token usage:   {response.usage}")

#### Multi-turn conversation

The Chat Completions format supports multi-turn conversations by including previous messages in the request.

In [ ]:
response = client.chat.completions.create(
    model="",
    messages=[
        {"role": "system", "content": "You are a helpful assistant that answers concisely."},
        {"role": "user", "content": "What is the largest planet in our solar system?"},
        {"role": "assistant", "content": "Jupiter is the largest planet in our solar system."},
        {"role": "user", "content": "How many moons does it have?"},
    ],
)

print(response.choices[0].message.content)

---

## Part 2: Inference component endpoint

[Inference components](https://docs.aws.amazon.com/sagemaker/latest/dg/inference-components.html) allow you to host multiple models on a single endpoint. Each inference component specifies which model to load and how many compute resources (memory, CPU cores, accelerators) it requires. This enables efficient hardware utilization when running multiple models that don't individually need a full instance.

### 2.1 Create the endpoint with an inference component

The key difference from a single model endpoint: the endpoint configuration does not reference a model. Instead, models are associated with inference components that are created separately and attached to the endpoint.

In [ ]:
IC_MODEL_NAME = f"openai-compat-ic-model-{TIMESTAMP}"
IC_ENDPOINT_CONFIG_NAME = f"openai-compat-ic-epc-{TIMESTAMP}"
IC_ENDPOINT_NAME = f"openai-compat-ic-ep-{TIMESTAMP}"
IC_NAME = f"openai-compat-ic-qwen3-4b-{TIMESTAMP}"

print(f"Model:           {IC_MODEL_NAME}")
print(f"Endpoint config: {IC_ENDPOINT_CONFIG_NAME}")
print(f"Endpoint:        {IC_ENDPOINT_NAME}")
print(f"Inference comp:  {IC_NAME}")

# Create the SageMaker model (same container and weights as before)
sagemaker_client.create_model(
    ModelName=IC_MODEL_NAME,
    ExecutionRoleArn=EXECUTION_ROLE,
    PrimaryContainer={
        "Image": VLLM_IMAGE,
        "Environment": {
            "HF_MODEL_ID": MODEL_HF_ID,
            "SM_VLLM_TENSOR_PARALLEL_SIZE": "1",
            "SM_VLLM_MAX_NUM_SEQS": "4",
            "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
            "SM_VLLM_TOOL_CALL_PARSER": "hermes",
            "SAGEMAKER_ENABLE_LOAD_AWARE": "1",
        },
    },
)
print(f"\nModel created: {IC_MODEL_NAME}")

In [ ]:
# Create endpoint configuration — note: no ModelName in the production variant
sagemaker_client.create_endpoint_config(
    EndpointConfigName=IC_ENDPOINT_CONFIG_NAME,
    ExecutionRoleArn=EXECUTION_ROLE,
    ProductionVariants=[
        {
            "VariantName": "variant1",
            "InstanceType": INSTANCE_TYPE,
            "InitialInstanceCount": 1
        }
    ],
)
print(f"Endpoint configuration created: {IC_ENDPOINT_CONFIG_NAME}")

In [ ]:
# Create the endpoint
sagemaker_client.create_endpoint(
    EndpointName=IC_ENDPOINT_NAME,
    EndpointConfigName=IC_ENDPOINT_CONFIG_NAME,
)
print(f"Endpoint creation initiated: {IC_ENDPOINT_NAME}")
print("Waiting for endpoint to reach InService status (this takes 5–10 minutes)...")

waiter = sagemaker_client.get_waiter("endpoint_in_service")
waiter.wait(
    EndpointName=IC_ENDPOINT_NAME,
    WaiterConfig={"Delay": 30, "MaxAttempts": 40},
)
print(f"Endpoint is InService: {IC_ENDPOINT_NAME}")

In [ ]:
# Create the inference component
sagemaker_client.create_inference_component(
    InferenceComponentName=IC_NAME,
    EndpointName=IC_ENDPOINT_NAME,
    VariantName="variant1",
    Specification={
        "ModelName": IC_MODEL_NAME,
        "ComputeResourceRequirements": {
            "MinMemoryRequiredInMb": 1024,
            "NumberOfCpuCoresRequired": 2,
            "NumberOfAcceleratorDevicesRequired": 1,
        },
    },
    RuntimeConfig={"CopyCount": 1},
)
print(f"Inference component creation initiated: {IC_NAME}")
print("Waiting for inference component to reach InService status...")

while True:
    desc = sagemaker_client.describe_inference_component(InferenceComponentName=IC_NAME)
    status = desc["InferenceComponentStatus"]
    if status == "InService":
        print(f"Inference component is InService: {IC_NAME}")
        break
    elif status == "Failed":
        raise RuntimeError(f"Inference component failed: {desc.get('FailureReason', 'unknown')}")
    time.sleep(30)

### 2.2 Invoke the inference component

To target a specific inference component, include its name in the URL path:

```
https://runtime.sagemaker.<REGION>.amazonaws.com/endpoints/<ENDPOINT>/inference-components/<IC_NAME>/openai/v1
```

This is the recommended approach because it makes the target model explicit in the URL, which aids debugging and request tracing.

In [ ]:
from openai import OpenAI
from sagemaker.core.token_generator import generate_token

ic_base_url = (
    f"https://runtime.sagemaker.{REGION}.amazonaws.com"
    f"/endpoints/{IC_ENDPOINT_NAME}/inference-components/{IC_NAME}/openai/v1"
)

ic_client = OpenAI(
    base_url=ic_base_url,
    api_key=generate_token(region=REGION),
)

print(f"Base URL: {ic_base_url}\n")

stream = ic_client.chat.completions.create(
    model="",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the benefits of using inference components in SageMaker?"},
    ],
    stream=True,
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="")
print()

### 2.3 Shared connection pool for multiple inference components

When you have multiple inference components on one endpoint, you can share a single `httpx` connection pool across all OpenAI client instances. Each client is a lightweight wrapper pointing to a different URL — the underlying TLS sessions and TCP connections are reused.

In [ ]:
import httpx
from openai import OpenAI
from sagemaker.core.token_generator import generate_token

# Shared connection pool — reused across all IC clients
shared_http = httpx.Client()

# In production, you would point each client at a different inference component.
# Here we demonstrate the pattern using a single IC.
client_a = OpenAI(
    base_url=(
        f"https://runtime.sagemaker.{REGION}.amazonaws.com"
        f"/endpoints/{IC_ENDPOINT_NAME}/inference-components/{IC_NAME}/openai/v1"
    ),
    api_key=generate_token(region=REGION),
    http_client=shared_http,
)

# Verify the shared pool works
response = client_a.chat.completions.create(
    model="",
    messages=[{"role": "user", "content": "What is 42 * 3? Reply with just the number."}],
)

print(f"Response: {response.choices[0].message.content}")
print(f"Connection pool active: shared_http is reusable across multiple IC clients")

---

## Part 3: Multi-agent workflow with Strands Agents

[Strands Agents](https://github.com/strands-agents/strands-agents-python) is an open-source SDK for building AI agents. Because it supports OpenAI-compatible model providers, you can use a SageMaker endpoint as the model backend.

The following example creates two agents backed by the same SageMaker endpoint — a **coder** that writes Python and a **reviewer** that evaluates the code. This demonstrates how a single endpoint can power multi-step AI workflows.

In [ ]:
from openai import AsyncOpenAI
from strands import Agent, tool
from strands.models.openai import OpenAIModel
from sagemaker.core.token_generator import generate_token

@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    return str(eval(expression))

# Create an AsyncOpenAI client pointing at the single model endpoint
strands_client = AsyncOpenAI(
    base_url=f"https://runtime.sagemaker.{REGION}.amazonaws.com/endpoints/{SME_ENDPOINT_NAME}/openai/v1",
    api_key=generate_token(region=REGION),
)

model = OpenAIModel(client=strands_client, model_id="", params={"temperature": 0.7})

In [ ]:
# Define the coder agent
coder = Agent(
    model=model,
    system_prompt=(
        "You are an expert Python developer. Write clean, well-documented "
        "Python code with type hints. Output ONLY the code, no explanation."
    ),
     tools=[calculator],
)

# Define the reviewer agent
reviewer = Agent(
    model=model,
    system_prompt=(
        "You are a senior code reviewer. Review Python code for correctness, "
        "performance, and PEP 8 style. Give a concise review with specific suggestions."
    ),
     tools=[calculator],
)

In [ ]:
# Run the workflow: coder writes code, reviewer reviews it
task = "Write a Python function that finds the longest palindromic substring in a string."

print("=" * 60)
print("CODER AGENT OUTPUT")
print("=" * 60)
code_output = coder(task)
print(code_output)

print("\n")
print("=" * 60)
print("REVIEWER AGENT OUTPUT")
print("=" * 60)
review_output = reviewer(f"Review this code:\n\n{code_output}")
print(review_output)

---

## Part 4: Auto-refreshing tokens for production

For long-running services, you can implement an `httpx.Auth` subclass that generates a fresh bearer token on every request. This avoids token expiry issues without requiring manual refresh logic in your application code.

In [ ]:
import httpx
from openai import OpenAI
from sagemaker.core.token_generator import generate_token


class SageMakerAuth(httpx.Auth):
    """Auto-refreshing bearer token auth for SageMaker OpenAI-compatible endpoints."""

    def __init__(self, region: str):
        self.region = region

    def auth_flow(self, request):
        request.headers["Authorization"] = f"Bearer {generate_token(region=self.region)}"
        yield request


# The httpx.Auth pattern generates a fresh token on every request
http_client = httpx.Client(auth=SageMakerAuth(region=REGION))

client = OpenAI(
    base_url=f"https://runtime.sagemaker.{REGION}.amazonaws.com/endpoints/{SME_ENDPOINT_NAME}/openai/v1",
    api_key="unused",  # Overridden by the httpx.Auth handler
    http_client=http_client,
)

# Verify with multiple requests — each gets a fresh token automatically
for i in range(3):
    response = client.chat.completions.create(
        model="",
        messages=[{"role": "user", "content": f"What is {(i + 1) * 11} + {(i + 1) * 7}? Reply with just the number."}],
    )
    print(f"Request {i + 1}: {response.choices[0].message.content.strip()}")

print("\nAll requests succeeded with auto-refreshed tokens.")

---

## Part 5: Cleanup

Delete all resources created in this notebook to stop incurring charges. Run this section when you are done testing.

> **Important:** SageMaker endpoints incur charges while they are in service, even if no requests are being made. Always delete endpoints when you are finished.

In [ ]:
# Delete inference component (must be deleted before its endpoint)
print(f"Deleting inference component: {IC_NAME}...")
sagemaker_client.delete_inference_component(InferenceComponentName=IC_NAME)

# Wait for inference component deletion to propagate
print("Waiting for inference component deletion to complete...")
time.sleep(60)

# Delete endpoints
print(f"Deleting endpoint: {IC_ENDPOINT_NAME}...")
sagemaker_client.delete_endpoint(EndpointName=IC_ENDPOINT_NAME)

print(f"Deleting endpoint: {SME_ENDPOINT_NAME}...")
sagemaker_client.delete_endpoint(EndpointName=SME_ENDPOINT_NAME)

# Wait for endpoint deletion to complete
print("Waiting for endpoint deletion to complete...")
time.sleep(60)

# Delete endpoint configurations
print(f"Deleting endpoint config: {IC_ENDPOINT_CONFIG_NAME}...")
sagemaker_client.delete_endpoint_config(EndpointConfigName=IC_ENDPOINT_CONFIG_NAME)

print(f"Deleting endpoint config: {SME_ENDPOINT_CONFIG_NAME}...")
sagemaker_client.delete_endpoint_config(EndpointConfigName=SME_ENDPOINT_CONFIG_NAME)

# Delete models
print(f"Deleting model: {IC_MODEL_NAME}...")
sagemaker_client.delete_model(ModelName=IC_MODEL_NAME)

print(f"Deleting model: {SME_MODEL_NAME}...")
sagemaker_client.delete_model(ModelName=SME_MODEL_NAME)

print("\nAll resources deleted successfully.")

---

## Summary

In this notebook, you:

1. Generated bearer tokens for authentication using `sagemaker.core.token_generator`
2. Deployed and invoked a **single model endpoint** with streaming and non-streaming requests
3. Deployed and invoked an **inference component** on a shared endpoint, including the shared connection pool pattern
4. Built a **multi-agent workflow** with Strands Agents using a SageMaker endpoint as the model backend
5. Implemented **auto-refreshing token authentication** for production use cases

For more information, see:
- [Amazon SageMaker Real-time Inference](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints.html)
- [SageMaker Inference Components](https://docs.aws.amazon.com/sagemaker/latest/dg/inference-components.html)
- [Strands Agents SDK](https://github.com/strands-agents/strands-agents-python)
- [OpenAI Python SDK](https://github.com/openai/openai-python)